# 01 · Document Parsing and Chunking
Demonstrates file parsing for all supported academic file types (PDF, PPTX, DOCX, XLSX, CSV, MD, HTML) and structure-aware chunking with metadata preservation.

In [ ]:
# ---- Shared setup (identical in every notebook) -------------------------------
import os, sys, json, pathlib, time
from dotenv import load_dotenv
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "sample_data"))

# Automatically load environment variables from rag_service/.env and repo-root .env
load_dotenv(ROOT / ".env", override=False)
load_dotenv(ROOT.parent / ".env", override=False)

# OFFLINE=True  -> deterministic hash embeddings + mock LLM + local Chroma. No keys, no network.
# OFFLINE=False -> REAL Gemini/Groq/OpenAI + Chroma Cloud from your .env (default when keys are present)
has_keys = bool(os.getenv("GEMINI_API_KEY") or os.getenv("GROQ_API_KEY") or os.getenv("OPENAI_API_KEY"))
OFFLINE = (os.getenv("NB_OFFLINE", "0" if has_keys else "1") == "1")
if OFFLINE:
    os.environ["RAG_OFFLINE"] = "1"
    os.environ["CHROMA_LOCAL_DIR"] = str(ROOT / "chroma_notebook")   # shared by all notebooks
else:
    os.environ["RAG_OFFLINE"] = "0"

from app.config import get_settings, reset_settings_cache
reset_settings_cache()
S = get_settings()
print("mode            :", "OFFLINE (mock)" if S.offline else "LIVE providers")
print("vector store    :", "Chroma Cloud" if S.use_chroma_cloud else f"local ({S.chroma_local_dir})")
print("embedding       :", "hash512" if S.offline else S.embedding_provider)
print("LLM providers   :", [p for p, k in (("gemini", S.gemini_key), ("groq", S.groq_key), ("openai", S.openai_key)) if k] or "none (offline mock)")

## 1. Create Synthetic Course Material

In [ ]:
import synthetic
files = synthetic.make_all()
for subj, fs in files.items():
    print(subj, {n: f"{len(b)//1024 or 1} KB" for n, b in fs.items()})

## 2. Page & Structure Aware Parsing

In [ ]:
from app.parsers import parse_bytes
secs = parse_bytes(files["BIO101"]["bio_notes.pdf"], "bio_notes.pdf")
for s in secs: print(f"page {s.page}:", s.text[:110].replace("\n"," "), "...")

## 3. Structure-Aware Chunking (Provenance, Headings, Stable IDs)

In [ ]:
from app.chunker import chunk_sections
chunks = chunk_sections(secs, "demo-doc", {"subject_code":"BIO101","is_active":True,"file_name":"bio_notes.pdf"})
print(len(chunks), "chunks created | sizes:", [len(c.page_content) for c in chunks])
print("First chunk metadata:")
print(json.dumps(chunks[0].metadata, indent=2))